# S6E4 Predicting Irrigation Need
## Score .96921

In [1]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold

pd.set_option("display.max_columns", 200)

In [2]:
PROJECT_ROOT = Path(".")
DATA_DIR = PROJECT_ROOT / "playground-series-s6e4"
ORIGINAL_PATH = PROJECT_ROOT / "original-data" / "irrigation_prediction.csv"
ANCHOR_SUB_PATH = PROJECT_ROOT / "score95787.csv"
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SUB_PATH = PROJECT_ROOT / "submission.csv"

TARGET = "Irrigation_Need"
ID_COL = "id"
SOURCE_COL = "_source"
BASE_SEED = 42
SEED_LIST = [42, 1337]
N_SPLITS = 3
STACKER_SPLITS = 5
USE_ORIGINAL_DATA = True
ORIGINAL_ROW_WEIGHT = 0.35
USE_ANCHOR_FALLBACK = False
FALLBACK_CONF_THRESHOLD = 0.60

if not TRAIN_PATH.exists() or not TEST_PATH.exists():
    raise FileNotFoundError(
        f"Expected data files under {DATA_DIR.resolve()}"
    )

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

train_df[SOURCE_COL] = "kaggle"

if USE_ORIGINAL_DATA and ORIGINAL_PATH.exists():
    original_df = pd.read_csv(ORIGINAL_PATH)
    original_df = original_df[[c for c in train_df.columns if c not in [ID_COL, SOURCE_COL]]]
    original_df[SOURCE_COL] = "original"
    train_df = pd.concat([train_df, original_df], axis=0, ignore_index=True)
    print("original shape:", original_df.shape)
else:
    print("original data disabled or missing")

print("train shape:", train_df.shape)
print("test shape :", test_df.shape)
train_df.head()

original shape: (10000, 21)
train shape: (640000, 22)
test shape : (270000, 20)


,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need,_source
0,0.0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low,kaggle
1,1.0,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low,kaggle
2,2.0,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,3.85,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low,kaggle
3,3.0,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,2.31,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium,kaggle
4,4.0,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,13.94,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low,kaggle


In [3]:
display(train_df[TARGET].value_counts(dropna=False))
print("\nTarget distribution (%):")
display((train_df[TARGET].value_counts(normalize=True) * 100).round(2))

missing_train = train_df.isna().mean().sort_values(ascending=False)
missing_test = test_df.isna().mean().sort_values(ascending=False)

print("\nTop missing-rate features (train):")
display(missing_train.head(10))
print("\nTop missing-rate features (test):")
display(missing_test.head(10))

Irrigation_Need
Low       375781
Medium    242874
High       21345
Name: count, dtype: int64


Target distribution (%):


Irrigation_Need
Low       58.72
Medium    37.95
High       3.34
Name: proportion, dtype: float64


Top missing-rate features (train):


id                         0.015625
Soil_Type                  0.000000
Soil_pH                    0.000000
Soil_Moisture              0.000000
Organic_Carbon             0.000000
Electrical_Conductivity    0.000000
Temperature_C              0.000000
Humidity                   0.000000
Rainfall_mm                0.000000
Sunlight_Hours             0.000000
dtype: float64


Top missing-rate features (test):


id                         0.0
Soil_Type                  0.0
Soil_pH                    0.0
Soil_Moisture              0.0
Organic_Carbon             0.0
Electrical_Conductivity    0.0
Temperature_C              0.0
Humidity                   0.0
Rainfall_mm                0.0
Sunlight_Hours             0.0
dtype: float64

In [4]:
y = train_df[TARGET].copy()
source_flag = train_df[SOURCE_COL].copy()
X = train_df.drop(columns=[TARGET]).copy()
X_test = test_df.copy()

feature_cols = [c for c in X.columns if c not in [ID_COL, SOURCE_COL]]
X = X[feature_cols]
X_test = X_test[feature_cols]

cat_cols = [c for c in feature_cols if X[c].dtype == "object"]
num_cols = [c for c in feature_cols if c not in cat_cols]

print(f"Total features: {len(feature_cols)}")
print(f"Categorical   : {len(cat_cols)} -> {cat_cols}")
print(f"Numerical     : {len(num_cols)}")
print("Source distribution:")
display(source_flag.value_counts())

Total features: 19
Categorical   : 8 -> ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']
Numerical     : 11
Source distribution:


_source
kaggle      630000
original     10000
Name: count, dtype: int64

In [5]:
run_start = time.perf_counter()
print("Starting ensemble CV run...")

classes = sorted(y.unique().tolist())
class_to_idx = {label: idx for idx, label in enumerate(classes)}
idx_to_class = {v: k for k, v in class_to_idx.items()}

y_idx = y.map(class_to_idx).astype(int)

class_counts = y.value_counts()
class_weight_label = {
    cls: len(y) / (len(classes) * class_counts[cls])
    for cls in classes
}
class_weight_idx = {
    class_to_idx[cls]: weight
    for cls, weight in class_weight_label.items()
}
source_weight_map = {
    "kaggle": 1.0,
    "original": ORIGINAL_ROW_WEIGHT,
}

oof_cat = np.zeros((len(X), len(classes)), dtype=np.float64)
oof_lgb = np.zeros((len(X), len(classes)), dtype=np.float64)
oof_xgb = np.zeros((len(X), len(classes)), dtype=np.float64)

test_cat = np.zeros((len(X_test), len(classes)), dtype=np.float64)
test_lgb = np.zeros((len(X_test), len(classes)), dtype=np.float64)
test_xgb = np.zeros((len(X_test), len(classes)), dtype=np.float64)

fold_scores = []

for seed_idx, seed in enumerate(SEED_LIST, start=1):
    seed_start = time.perf_counter()
    print(f"\n######## Seed {seed_idx}/{len(SEED_LIST)} (seed={seed}) ########")

    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)

    seed_test_cat = np.zeros((len(X_test), len(classes)), dtype=np.float64)
    seed_test_lgb = np.zeros((len(X_test), len(classes)), dtype=np.float64)
    seed_test_xgb = np.zeros((len(X_test), len(classes)), dtype=np.float64)

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
        fold_start = time.perf_counter()
        print(f"\n=== Seed {seed} | Fold {fold}/{N_SPLITS} started ===")

        X_train, X_valid = X.iloc[train_idx].copy(), X.iloc[valid_idx].copy()
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
        y_train_idx, y_valid_idx = y_idx.iloc[train_idx], y_idx.iloc[valid_idx]
        source_train = source_flag.iloc[train_idx]

        for col in cat_cols:
            X_train[col] = X_train[col].astype("category")
            X_valid[col] = X_valid[col].astype("category")

        X_test_fold = X_test.copy()
        for col in cat_cols:
            X_test_fold[col] = X_test_fold[col].astype("category")

        sample_weight = (
            y_train.map(class_weight_label).values
            * source_train.map(source_weight_map).values
        )

        cat_model = CatBoostClassifier(
            iterations=700,
            learning_rate=0.06,
            depth=7,
            loss_function="MultiClass",
            eval_metric="TotalF1",
            random_seed=seed + fold,
            verbose=100,
            thread_count=-1,
            class_weights=class_weight_label,
        )

        lgb_model = LGBMClassifier(
            n_estimators=800,
            learning_rate=0.04,
            num_leaves=63,
            min_child_samples=40,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_lambda=1.0,
            objective="multiclass",
            random_state=seed + fold,
            class_weight=class_weight_idx,
            verbosity=-1,
        )

        xgb_model = XGBClassifier(
            n_estimators=800,
            learning_rate=0.04,
            max_depth=7,
            min_child_weight=3,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_lambda=1.0,
            objective="multi:softprob",
            num_class=len(classes),
            eval_metric="mlogloss",
            tree_method="hist",
            enable_categorical=True,
            random_state=seed + fold,
            n_jobs=-1,
        )

        print(f"Seed {seed} Fold {fold}: training CatBoost...")
        cat_model.fit(
            X_train,
            y_train,
            sample_weight=sample_weight,
            cat_features=cat_cols,
            eval_set=(X_valid, y_valid),
            use_best_model=True,
            early_stopping_rounds=80,
        )

        print(f"Seed {seed} Fold {fold}: training LightGBM...")
        lgb_model.fit(
            X_train,
            y_train_idx,
            sample_weight=sample_weight,
            eval_set=[(X_valid, y_valid_idx)],
            eval_metric="multi_logloss",
            callbacks=[early_stopping(stopping_rounds=80), log_evaluation(period=0)],
        )

        print(f"Seed {seed} Fold {fold}: training XGBoost...")
        xgb_model.fit(
            X_train,
            y_train_idx,
            eval_set=[(X_valid, y_valid_idx)],
            sample_weight=sample_weight,
            verbose=False,
        )
        print(f"Seed {seed} Fold {fold}: model training complete")

        cat_valid = cat_model.predict_proba(X_valid)
        lgb_valid = lgb_model.predict_proba(X_valid)
        xgb_valid = xgb_model.predict_proba(X_valid)

        oof_cat[valid_idx] += cat_valid / len(SEED_LIST)
        oof_lgb[valid_idx] += lgb_valid / len(SEED_LIST)
        oof_xgb[valid_idx] += xgb_valid / len(SEED_LIST)

        cat_pred = np.argmax(cat_valid, axis=1)
        lgb_pred = np.argmax(lgb_valid, axis=1)
        xgb_pred = np.argmax(xgb_valid, axis=1)

        cat_score = balanced_accuracy_score(y_valid, [idx_to_class[i] for i in cat_pred])
        lgb_score = balanced_accuracy_score(y_valid, [idx_to_class[i] for i in lgb_pred])
        xgb_score = balanced_accuracy_score(y_valid, [idx_to_class[i] for i in xgb_pred])

        print(
            f"Seed {seed} Fold {fold}: cat={cat_score:.6f} lgb={lgb_score:.6f} xgb={xgb_score:.6f}"
        )

        blend_equal = (cat_valid + lgb_valid + xgb_valid) / 3.0
        blend_pred = np.argmax(blend_equal, axis=1)
        fold_score = balanced_accuracy_score(y_valid, [idx_to_class[i] for i in blend_pred])
        fold_scores.append(fold_score)
        fold_minutes = (time.perf_counter() - fold_start) / 60.0
        print(f"Seed {seed} Fold {fold}: equal-blend balanced_accuracy = {fold_score:.6f}")
        print(f"Seed {seed} Fold {fold}: completed in {fold_minutes:.2f} minutes")

        seed_test_cat += cat_model.predict_proba(X_test_fold)
        seed_test_lgb += lgb_model.predict_proba(X_test_fold)
        seed_test_xgb += xgb_model.predict_proba(X_test_fold)

    test_cat += seed_test_cat / N_SPLITS
    test_lgb += seed_test_lgb / N_SPLITS
    test_xgb += seed_test_xgb / N_SPLITS

    print(
        f"Seed {seed} finished in {(time.perf_counter() - seed_start) / 60.0:.2f} minutes"
    )

test_cat /= len(SEED_LIST)
test_lgb /= len(SEED_LIST)
test_xgb /= len(SEED_LIST)

print("\nAll folds done. Starting cross-fitted OOF stacking...")

meta_oof = np.hstack([oof_cat, oof_lgb, oof_xgb])
meta_test = np.hstack([test_cat, test_lgb, test_xgb])

best_stack_score = -1.0
best_stack_c = None
best_stack_proba_oof = None
best_stack_proba_test = None

stack_cv = StratifiedKFold(n_splits=STACKER_SPLITS, shuffle=True, random_state=BASE_SEED)

for c in [0.25, 0.5, 1.0, 2.0, 4.0]:
    print(f"Stacking search: cross-fitting LogisticRegression C={c}")

    cv_meta_oof = np.zeros((len(meta_oof), len(classes)), dtype=np.float64)
    cv_meta_test = np.zeros((len(meta_test), len(classes)), dtype=np.float64)

    for stack_fold, (st_tr_idx, st_va_idx) in enumerate(stack_cv.split(meta_oof, y_idx), start=1):
        X_st_tr, X_st_va = meta_oof[st_tr_idx], meta_oof[st_va_idx]
        y_st_tr = y_idx.iloc[st_tr_idx]

        stacker = LogisticRegression(
            C=c,
            max_iter=400,
            multi_class="multinomial",
            solver="lbfgs",
            class_weight="balanced",
        )
        stacker.fit(X_st_tr, y_st_tr)

        cv_meta_oof[st_va_idx] = stacker.predict_proba(X_st_va)
        cv_meta_test += stacker.predict_proba(meta_test)

        if stack_fold % 2 == 0:
            print(f"Stacking C={c}: finished meta-fold {stack_fold}/{STACKER_SPLITS}")

    cv_meta_test /= STACKER_SPLITS

    pred_oof = np.argmax(cv_meta_oof, axis=1)
    pred_oof_labels = [idx_to_class[i] for i in pred_oof]
    score = balanced_accuracy_score(y, pred_oof_labels)
    print(f"Stacking C={c}: cross-fitted OOF balanced_accuracy = {score:.6f}")

    if score > best_stack_score:
        best_stack_score = score
        best_stack_c = c
        best_stack_proba_oof = cv_meta_oof
        best_stack_proba_test = cv_meta_test

print(f"Best stacker C: {best_stack_c} with OOF score {best_stack_score:.6f}")
print("Stacking done. Starting class-bias tuning...")

best_bias_score = -1.0
best_bias = np.zeros(len(classes), dtype=np.float64)

bias_checks = 0
for b_low in np.arange(-0.06, 0.061, 0.01):
    for b_med in np.arange(-0.06, 0.061, 0.01):
        for b_high in np.arange(-0.06, 0.061, 0.01):
            bias = np.array([b_low, b_med, b_high], dtype=np.float64)
            biased_oof = best_stack_proba_oof + bias
            pred_idx = np.argmax(biased_oof, axis=1)
            pred_labels = [idx_to_class[i] for i in pred_idx]
            score = balanced_accuracy_score(y, pred_labels)
            bias_checks += 1
            if bias_checks % 200 == 0:
                print(f"Bias search progress: checked {bias_checks} combos")
            if score > best_bias_score:
                best_bias_score = score
                best_bias = bias.copy()

oof_best = best_stack_proba_oof + best_bias
oof_best_pred = np.argmax(oof_best, axis=1)
oof_best_pred = [idx_to_class[i] for i in oof_best_pred]

test_proba = best_stack_proba_test + best_bias

cv_score = balanced_accuracy_score(y, oof_best_pred)
print("\nCV balanced accuracy (OOF):", round(cv_score, 6))
print("Fold equal-blend scores:", [round(s, 6) for s in fold_scores])
print("Mean equal-blend score:", round(float(np.mean(fold_scores)), 6))
print(f"Best OOF score from stacking search: {best_stack_score:.6f}")
bias_text = ", ".join(
    [f"{cls}={offset:+.3f}" for cls, offset in zip(classes, best_bias)]
)
print(f"Best class bias offsets: {bias_text}")
print(f"Best OOF score after bias tuning: {best_bias_score:.6f}")
print(f"Total bias combos checked: {bias_checks}")
print(f"Total run time: {(time.perf_counter() - run_start) / 60.0:.2f} minutes")

Starting ensemble CV run...

######## Seed 1/2 (seed=42) ########

=== Seed 42 | Fold 1/3 started ===
Seed 42 Fold 1: training CatBoost...
0:	learn: 0.9546693	test: 0.8096672	best: 0.8096672 (0)	total: 813ms	remaining: 9m 28s
100:	learn: 0.9765345	test: 0.9183051	best: 0.9183714 (99)	total: 55s	remaining: 5m 26s
200:	learn: 0.9787932	test: 0.9229771	best: 0.9229917 (199)	total: 1m 46s	remaining: 4m 24s
300:	learn: 0.9818450	test: 0.9297467	best: 0.9297467 (300)	total: 2m 47s	remaining: 3m 41s
400:	learn: 0.9844878	test: 0.9352488	best: 0.9352488 (400)	total: 3m 52s	remaining: 2m 53s
500:	learn: 0.9866411	test: 0.9401863	best: 0.9401930 (499)	total: 4m 53s	remaining: 1m 56s
600:	learn: 0.9882870	test: 0.9449130	best: 0.9449130 (600)	total: 5m 56s	remaining: 58.7s
699:	learn: 0.9896303	test: 0.9494103	best: 0.9494103 (699)	total: 7m 5s	remaining: 0us

bestTest = 0.9494103085
bestIteration = 699

Seed 42 Fold 1: training LightGBM...


c:\Users\ol1v3_7dwns5u\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\ol1v3_7dwns5u\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "c:\Users\ol1v3_7dwns5u\AppData\Local\Programs\Python\Python313\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\ol1v3_7dwns5u\AppData\Local\Programs\Python\Python3

Training until validation scores don't improve for 80 rounds
Did not meet early stopping. Best iteration is:
[800]	valid_0's multi_logloss: 0.0659345
Seed 42 Fold 1: training XGBoost...
Seed 42 Fold 1: model training complete
Seed 42 Fold 1: cat=0.949866 lgb=0.970276 xgb=0.968913
Seed 42 Fold 1: equal-blend balanced_accuracy = 0.971230
Seed 42 Fold 1: completed in 8.83 minutes

=== Seed 42 | Fold 2/3 started ===
Seed 42 Fold 2: training CatBoost...
0:	learn: 0.9551200	test: 0.8044904	best: 0.8044904 (0)	total: 644ms	remaining: 7m 30s
100:	learn: 0.9772348	test: 0.9208140	best: 0.9208140 (100)	total: 59.1s	remaining: 5m 50s
200:	learn: 0.9792550	test: 0.9260859	best: 0.9260859 (200)	total: 1m 59s	remaining: 4m 57s
300:	learn: 0.9828920	test: 0.9327024	best: 0.9327024 (300)	total: 3m 7s	remaining: 4m 8s
400:	learn: 0.9849582	test: 0.9382868	best: 0.9382868 (400)	total: 4m 12s	remaining: 3m 8s
500:	learn: 0.9872911	test: 0.9430825	best: 0.9430825 (500)	total: 5m 20s	remaining: 2m 7s
600:	

C:\Users\ol1v3_7dwns5u\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
C:\Users\ol1v3_7dwns5u\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Stacking C=0.25: finished meta-fold 2/5


...

Stacking C=4.0: finished meta-fold 2/5


C:\Users\ol1v3_7dwns5u\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
C:\Users\ol1v3_7dwns5u\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Stacking C=4.0: finished meta-fold 4/5


C:\Users\ol1v3_7dwns5u\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Stacking C=4.0: cross-fitted OOF balanced_accuracy = 0.971574
Best stacker C: 0.25 with OOF score 0.971612
Stacking done. Starting class-bias tuning...
Bias search progress: checked 200 combos
Bias search progress: checked 400 combos
Bias search progress: checked 600 combos
Bias search progress: checked 800 combos
Bias search progress: checked 1000 combos
Bias search progress: checked 1200 combos
Bias search progress: checked 1400 combos
Bias search progress: checked 1600 combos
Bias search progress: checked 1800 combos
Bias search progress: checked 2000 combos

CV balanced accuracy (OOF): 0.971648
Fold equal-blend scores: [0.97123, 0.971257, 0.969055, 0.97042, 0.970076, 0.971751]
Mean equal-blend score: 0.970631
Best OOF score from stacking search: 0.971612
Best class bias offsets: High=-0.060, Low=+0.060, Medium=-0.050
Best OOF score after bias tuning: 0.971648
Total bias combos checked: 2197
Total run time: 90.66 minutes


In [6]:
test_conf = np.max(test_proba, axis=1)
test_pred_idx = np.argmax(test_proba, axis=1)
test_pred = pd.Series([idx_to_class[i] for i in test_pred_idx], index=test_df.index)

if USE_ANCHOR_FALLBACK and ANCHOR_SUB_PATH.exists():
    anchor_df = pd.read_csv(ANCHOR_SUB_PATH)
    anchor_df = anchor_df[[ID_COL, TARGET]].copy()
    anchor_map = anchor_df.set_index(ID_COL)[TARGET]
    anchor_labels = test_df[ID_COL].map(anchor_map)

    if anchor_labels.isna().any():
        raise ValueError("Anchor submission is missing ids from test set")

    use_anchor_mask = test_conf < FALLBACK_CONF_THRESHOLD
    test_pred.loc[use_anchor_mask] = anchor_labels.loc[use_anchor_mask]
    print(
        f"Anchor fallback applied to {int(use_anchor_mask.sum())} rows "
        f"(threshold={FALLBACK_CONF_THRESHOLD})"
    )
else:
    print("Anchor fallback disabled or anchor file missing")

submission = pd.DataFrame({
    ID_COL: test_df[ID_COL],
    TARGET: test_pred.values,
})
submission.to_csv(SUB_PATH, index=False)

print("Saved:", SUB_PATH.resolve())
submission.head()

Anchor fallback disabled or anchor file missing
Saved: C:\Users\ol1v3_7dwns5u\OneDrive\Desktop\ACTIVE\s6e4-predicting-irrigation-need\submission.csv


,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low
